# 03 · Clean, decide, reconcile

**Produces:** `data/clean/bookings_clean.csv.gz`, `data/clean/collections_clean.csv.gz`, `data/clean/patients_clean.csv.gz`, `data/clean/invoices_clean.csv.gz`, `data/clean/payments_clean.csv.gz`, and rows in the decisions log for every rule applied.

**Needs:** notebook 02 run.

Every cleaning act here is a decision with a reason, a row count and an effect on the headline. Input equals clean plus rejected after every step. Joins are checked before and after on row count, and a join that grows is explained or reverted.

## CONFIG

In [14]:
from kalpa import ai

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

2.3.3 2.3.5


In [2]:
bookings = pd.read_csv(CONFIG["clean"] / "bookings.csv.gz", parse_dates=["created_at", "slot_at"], dtype={"booking_id": "string", "patient_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
HEADLINE = lambda df: f"{len(df):,} bookings, {int(df['n_tests'].sum()):,} tests"
n0 = len(bookings)
print(HEADLINE(bookings))

564,879 bookings, 1,307,260 tests


## 1 · DO: the identity rule for a booking

Nova records a rescheduled visit as a new booking that points at the old one, and marks the old one `rescheduled`. Legacy overwrote the slot in place. If the question is "how many visits happened", a rescheduled pair is one visit. Write the rule, apply it, log it.

In [ ]:
# status_at_export is NA for every legacy row. A nullable boolean treats NA as False only if you say so; left alone,
# ~NA is NA and the legacy rows vanish from the filter. The trap is worth seeing once, so it is stated here.
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
kept = bookings[~is_superseded].copy()
rejected = bookings[is_superseded]
checks.reconciles("drop superseded nova records", n0, len(kept), len(rejected))
print("after:", HEADLINE(kept))
# saved for the decisions log (section 5); bookings is overwritten below, so a re-run of this cell alone would see 0
R1 = {"before": HEADLINE(bookings), "after": HEADLINE(kept), "rows": len(rejected),
      "bookings_in": n0, "bookings_out": len(kept),
      "tests_in": int(bookings["n_tests"].sum()), "tests_out": int(kept["n_tests"].sum())}
bookings = kept

> **Interview angle.** What is `pd.NA`, and what happens to `~series.eq('x')` when the series holds NA?
>
> *What a good answer covers:* pd.NA is the missing marker for nullable dtypes and propagates through comparisons, so eq gives NA, the negation stays NA, and a boolean filter drops those rows silently. fillna(False) before negating, or compare with notna() explicitly.

> **Interview angle.** Two systems record the same business event in different shapes. What is the general method for making them comparable, and what is the trap?
>
> *What a good answer covers:* Trace one real event through both systems by hand and write down how each represents it; then write a rule that maps both to one definition of the event; then apply it and check the count before and after. The trap is comparing totals without the rule and attributing the difference to the business.

## 2 · DO: collections, double scans, and the join that must not grow

In [5]:
coll_files = sorted((CONFIG["raw"] / "collections").glob("*.csv.gz"))
coll = loaders.read_many(coll_files, "legacy", workers=1) if False else pd.concat([pd.read_csv(p, dtype={"booking_id": "string"}) for p in coll_files], ignore_index=True)
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")
n_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")
coll_clean = coll[~dups].copy()
checks.reconciles("collections exact duplicates (double scans)", n_in, len(coll_clean), int(dups.sum()))
checks.check("one collection row per booking", coll_clean["booking_id"].is_unique, f"{(~coll_clean['booking_id'].is_unique)} duplicates remain" if not coll_clean["booking_id"].is_unique else "")

PASS  collections exact duplicates (double scans): input = kept + rejected   552,492 = 549,598 + 2,894
PASS  one collection row per booking


True

In [6]:
# LEFT join bookings to collections. The row count must not grow. validate= makes pandas raise rather than lie.
try:
    joined = bookings.merge(coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
except pd.errors.MergeError as err:
    print("MergeError:", err)
    joined = bookings.merge(coll_clean.drop_duplicates("booking_id")[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
checks.join_did_not_fan_out("bookings LEFT JOIN collections", bookings, joined, "booking_id")
joined["_merge"].value_counts()

PASS  bookings LEFT JOIN collections: join kept the left row count   left 563,380, joined 563,380, rows on repeated keys 0


_merge
both          545111
left_only      18269
right_only         0
Name: count, dtype: int64

### SEE: what `left_only` means

A booking with no collection row is a cancellation (the phlebotomy app does not record cancelled bookings) or a booking in the future at export time. Decide which, and write it down. A row that is `left_only` and dated months ago is a question for the data team, and a caveat on your slide.

In [7]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
skew = joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)
print("collected more than a day before the slot (clock skew?):", int(skew.sum()))
joined.loc[skew, "collected_at"] = pd.NaT       # keep the row, blank the timestamp, flag it
joined["flag_clock_skew"] = skew
bookings = joined.drop(columns="_merge")

collected more than a day before the slot (clock skew?): 936


> **Interview angle.** Explain `how='left'`, `validate='one_to_one'` and `indicator=True` to someone who has only used SQL.
>
> *What a good answer covers:* LEFT keeps every left row whether or not it matches, like SQL LEFT JOIN. validate asserts the key's cardinality on each side and raises if it is violated, which SQL does not do for you. indicator adds a column saying which side each row came from, which is the SQL anti-join check (WHERE right.key IS NULL) made explicit.

## 3 · DO: patients, and the duplicate that looks different

In [8]:
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz", dtype="string")
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
grp = pat.groupby(["phone_norm", "name_norm", "dob"]).ngroup()
pat["person_key"] = grp
dup_mask = pat.duplicated(subset=["person_key"], keep="first")
print(f"{len(pat):,} patient rows, {pat['person_key'].nunique():,} distinct people by phone+name+dob, {int(dup_mask.sum()):,} duplicate rows")
bad_dob = pat["dob"].eq("1900-01-01")
print("placeholder dob rows:", int(bad_dob.sum()))
pat_clean = pat[~dup_mask].copy()
checks.reconciles("patients de-duplicated", len(pat), len(pat_clean), int(dup_mask.sum()))
# map every duplicate id to its surviving id so bookings can be re-keyed to people, not to ids
id_map = pat.merge(pat_clean[["person_key", "patient_id"]].rename(columns={"patient_id": "person_id"}), on="person_key")[["patient_id", "person_id"]]
bookings = bookings.merge(id_map, on="patient_id", how="left", validate="many_to_one")
checks.check("every booking patient id maps to a person", bool(bookings["person_id"].notna().all()), int(bookings["person_id"].isna().sum()))

309,217 patient rows, 303,153 distinct people by phone+name+dob, 6,064 duplicate rows
placeholder dob rows: 1492
PASS  patients de-duplicated: input = kept + rejected   309,217 = 303,153 + 6,064
PASS  every booking patient id maps to a person   0


True

### The decision here is not technical

Merging two patient rows says two records are one human. For a growth analysis the cost of being wrong is a slightly wrong frequency figure. For a results system it would be a wrong report sent to the wrong person. Same rule, different stakes. Write the stakes in the decisions log beside the rule.

## 4 · DO: invoices and payments, two id families

In [10]:
inv = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "invoices").glob("*.csv.gz"))], ignore_index=True)
inv["amount_rs"] = inv["amount"].str.replace(",", "", regex=False).astype(float)
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
pay = pd.concat([pd.read_csv(p, dtype={"notes_invoice_ref": "string", "settlement_id": "string"}) for p in sorted((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))], ignore_index=True)
pay["amount_rs"] = pay["amount_paise"] / 100
pay["created_at"] = pd.to_datetime(pay["created_at_epoch"], unit="s") + pd.Timedelta(hours=5, minutes=30)
pay["invoice_id"] = loaders.normalise_invoice_ref(pay["notes_invoice_ref"])
print(f"invoices {len(inv):,} for Rs {inv['amount_rs'].sum():,.0f};  payment rows {len(pay):,} ({(pay.status=='captured').sum():,} captured, {(pay.status=='failed').sum():,} failed)")
checks.check("invoice ids unique", inv["invoice_id"].is_unique)

invoices 470,776 for Rs 737,020,069;  payment rows 473,507 (455,469 captured, 18,038 failed)
PASS  invoice ids unique


True

In [11]:
captured = pay[pay["status"] == "captured"]
m = inv.merge(captured[["invoice_id", "payment_id", "amount_rs", "created_at"]].rename(columns={"amount_rs": "paid_rs"}), on="invoice_id", how="outer", indicator=True, validate="one_to_many")
print(m["_merge"].value_counts())
multi = m[m["_merge"] == "both"].groupby("invoice_id").size()
print("invoices with more than one captured payment:", int((multi > 1).sum()))

_merge
both          450881
left_only      21820
right_only      4588
Name: count, dtype: int64
invoices with more than one captured payment: 1925


> **Interview angle.** Your LEFT join of invoices to payments grew the row count. Name three possible causes and the check that separates them.
>
> *What a good answer covers:* Retries (same invoice, a failed and a captured row): filter by status before joining. Genuine double charges (two captured rows): group by invoice and count captured. A many-to-many on a non-unique key such as amount and date: check key uniqueness on the right side with validate='one_to_many' or a groupby count before the join.

## 5 · SUM: write the clean tables and log every rule

In [12]:
bookings.to_csv(CONFIG["clean"] / "bookings_clean.csv.gz", index=False, compression="gzip")
coll_clean.to_csv(CONFIG["clean"] / "collections_clean.csv.gz", index=False, compression="gzip")
pat_clean.to_csv(CONFIG["clean"] / "patients_clean.csv.gz", index=False, compression="gzip")
inv.to_csv(CONFIG["clean"] / "invoices_clean.csv.gz", index=False, compression="gzip")
pay.to_csv(CONFIG["clean"] / "payments_clean.csv.gz", index=False, compression="gzip")
print("headline after cleaning:", HEADLINE(bookings))
checks.summary()

headline after cleaning: 563,380 bookings, 1,303,719 tests
7 of 8 checks passed


,check,passed,detail
0,drop superseded nova records: input = kept + r...,True,"564,879 = 563,380 + 1,499"
1,drop superseded nova records: input = kept + r...,False,"564,879 = 563,380 + 0"
2,collections exact duplicates (double scans): i...,True,"552,492 = 549,598 + 2,894"
3,one collection row per booking,True,
4,bookings LEFT JOIN collections: join kept the ...,True,"left 563,380, joined 563,380, rows on repeated..."
5,patients de-duplicated: input = kept + rejected,True,"309,217 = 303,153 + 6,064"
6,every booking patient id maps to a person,True,0
7,invoice ids unique,True,


In [13]:
logs.decision(
    table="bookings",
    found="Nova adds a new booking on reschedule and marks the old one `rescheduled`. Legacy edits the slot in place.",
    decision="Drop the superseded old record",
    reason="One visit = one booking. Makes nova and legacy comparable.",
    rows_affected=1499,
    effect_on_headline="564,879 -> 563,380 bookings (-0.27%); 1,307,260 -> 1,303,719 tests",
    reversible="yes",
    owner="Rohit",
    n=1,
)
logs.decision(
    table="collections",
    found="Same booking, status and time scanned twice",
    decision="Drop exact duplicates, keep first",
    reason="A double scan is one collection",
    rows_affected=2894,
    effect_on_headline="none on bookings; collections 552,492 -> 549,598",
    reversible="yes",
    owner="Rohit",
    n=2,
)
logs.decision(
    table="bookings",
    found="18,269 bookings have no collection row",
    decision="Keep, set status `cancelled_or_missing`",
    reason="The app does not record cancels, so cancelled and missing cannot be told apart. Ask the data team.",
    rows_affected=18269,
    effect_on_headline="none (rows kept), but the headline still includes them",
    reversible="yes",
    owner="Rohit",
    n=3,
)
logs.decision(
    table="bookings",
    found="936 collections are stamped over 1 day before the slot (clock skew?)",
    decision="Keep the row, blank `collected_at`, set `flag_clock_skew`",
    reason="The time is wrong but the booking is real",
    rows_affected=936,
    effect_on_headline="none",
    reversible="yes",
    owner="Rohit",
    n=4,
)
logs.decision(
    table="patients",
    found="6,064 of 309,217 rows share phone+name+dob with an earlier row; 1,492 have placeholder dob 1900-01-01",
    decision="Merge into one person (keep first), re-key bookings to the person",
    reason="Growth analysis only. A wrong merge slightly skews frequency. No reports go to patients.",
    rows_affected=6064,
    effect_on_headline="309,217 -> 303,153 patients (-2.0%); bookings unchanged",
    reversible="yes",
    owner="Rohit",
    n=5,
)
logs.decision(
    table="invoices, payments",
    found="1,925 invoices have 2+ captured payments, 21,820 have none, 4,588 payments have no invoice",
    decision="Keep all, no rule yet",
    reason="Needs finance input before dropping anything",
    rows_affected=0,
    effect_on_headline="none yet. Rs 737,020,069 invoiced",
    reversible="yes",
    owner="Rohit",
    n=6,
)


In [15]:
ai.log(asked="I asked it generate the decision logs",
       did="Generate the decision logs for the data cleaning and reconciliation steps, summarizing the findings, decisions made, reasons, rows affected, effects on headline metrics and reversibility.",
       who="Rohit ", tool="Claude Code", n=2)

## Before you move on

- List every rule you applied, in order, with the headline number before and after each. That list is `docs/decisions-log.md`; if it is not there yet, stop and write it.
- Which rule moved the headline most? That rule is on slide 5.
- Which rule would the finance controller challenge first?